### **Table of Content**
- Chapter 1.0: Introduction to Prompt Engineering

### **Key Highlights**
-
-
-

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv  # Import the specific functions from dotenv
from openai import OpenAI, RateLimitError  # Import the specific error & OpenAI

#### **1.0 Introduction to Prompt Engineering**

Prompt engineering refers to crafting effective prompts to guide the language model towards the intended response. By refining your prompts, you can achieve better results and guide the model towards generating more accurate and useful responses. 

In [2]:
# find_dotenv() will automatically climb up from 'developer_path' 
# to the root folder to find your .env file flawlessly.
load_dotenv(find_dotenv())

# Retrieve the key
api_key = os.getenv("OPENAI_API_KEY")

# Safety check to make sure it loaded
if not api_key:
    raise ValueError("API Key is still missing! Double-check the variable name inside your .env file.")

# Initialize the client
client = OpenAI(api_key=api_key)
print("Connected successfully! OpenAI client is ready.")

Connected successfully! OpenAI client is ready.


In [ ]:
# Creating a function for the prompt
def get_response(prompt): # param to pass user input/instructions
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=100,
        messages=[
            {"role": "system", "content": "You are a helpful general assistant."}, # or create var prompt to insert user input/instructions
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content.strip()

#### **2.0 Key Principles of Prompt Engineering**
- Suitable verbs
- Clear & precise prompts
- Well structured delimited prompts


In [ ]:
prompt = "What is the capital of France? ```story```"
get_response(prompt)

print("")

#### **3.0 Structured Outputs & Conditional Prompts**


In [ ]:
# Table formatting
prompt = "Generate a table containing 5 rows and 3 columns with random data."
# List formatting
prompt = "Generate a list of 5 random items."
# Structured data formatting
prompt = "Generate a JSON object with 3 key-value pairs representing a person's name, age, and occupation."

# Custom output
text = "Once a upon a time in a land far, far away [...]"
instructions = "You will be provided a text delimited by triple backticks. Your task is to summarize the text in one sentence."
output_format = """
Provide the summary in the following format: 
- Text: <the original text>
- Title: <a title for the text>
- Summary: <the one-sentence summary>
"""
prompt = instructions + output_format + f"```{text}```"

# Conditional prompting
text = ""
prompt = f"""
You will be provided with a text delimited by triple backticks. If the text is written in English,
suugest title for it. Otherwise, respond with "The text is not in English."

```{text}```
"""


response = get_response(prompt)
print(response)


#### **5.0 Few-shot Prompting**
- Model provided with examples (question-answer pair)
- Number of examples: zero (zero-shot prompting), one (one-shot prompting), more than one (few shot prompting)


In [ ]:
# zero-shot prompting
prompt = "What is prompt engineering?"

# one-shot prompting
prompt = "What is prompt engineering? Here's an example of a good answer: Prompt engineering is the process of designing and refining prompts to effectively communicate with AI models, ensuring they understand the task and provide accurate responses."

# few-shot prompting
prompt = """
Text: I love programming in Python. -> Classsification: Positive
Text: I dislike bugs in my code. -> Classification: Negative
Text: I like to play video games. -> Classification: Positive
Text: I find debugging frustrating. -> Classification: 
"""

In [ ]:
# Creating the chat completion with the messages array to include the system, user, and assistant roles
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role":"user", 
            "content":"I love programming in Python." 
        },

        {
            "role":"assistant", 
            "content":"positive"
        },

        {
            "role":"use", 
            "content":"I dislike bugs in my code."
        },

        {
            "role":"user", 
            "content":prompt
        }
    
    ],
    temperature=0.7, # controls randomness of the output
    max_tokens=100, # controls the length of the output
)

print(response.choices[0].message.content.strip())

#### **6.0 Multi-step Prompting**
- This is used for **Sequential & Cognitive tasks**
- Break down an end goal into series of steps
- Model goes through each step to give final output
- Incorporate steps inside the prompt

In [ ]:
# Single-step prompt: Writing a blog
prompt = "Compose a travel blog"

# Multi-step prompt: Writing a blog with specific sections
prompt = """Compose a travel blog with the following sections:
1. Introduction
2. Itinerary
3. Recommendations
4. Conclusion
"""

Analyzing solutions correctness of model output

In [ ]:
text = "Hello, how are you?"

# Single-step prompting
prompt = f"""
Determine if the following text is in English or not. Respond with "The text is in English." or "The text is not in English."
```{text}```
"""

# Multi-step prompting
prompt = f"""
Determine if the following text is in English or not. Respond with "The text is in English." or "The text is not in English."
Step 1: Analyze the text to identify the language.
Step 2: Based on the analysis, determine if the text is in English or not.
Text: ```{text}```
"""

#### **7.0 Chain-of-thought & Self-consistency Prompting**
- **Chain-of-thought prompting:**
    - Requires LLMs to provide reasoning steps (thoughts) before giving answer
    - Complex reasoning tasks
    - Reduce model errors


In [ ]:
std_prompt = """"
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: The answer is
"""

chain_prompt = """
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: Let's break down the problem step by step.
"""

# Chain-of-thought prompting with few-shots
example = """
Q: You have 10 apples. You eat 2 apples and then buy 4 more apples. How many apples do you have now?
A: From 10 exisitng apples, minus 2 eaten and then add 4 more, the total number of apples is 16.
"""

question = """
Q: You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?
A: 
"""

prompt = example + question

- **Self-consistency Prompting:**
    - Generate multiple chain-of-thoughts by prompting model several times
    - Majority vote to obtain final output
    - Define by multiple prompts or prompt generating multiple responses

In [ ]:
self_consistency_instructions = """
Imagine three completely independent experts who reason differently are answering this question.
The final answer is obtained by majority vote.
The question is:
"""

problem_to_solve = "You start with 15 books. At the bookstore, you buy 5 more books and then give away 3 books to a friend. How many books do you have now?"

prompt = self_consistency_instructions + problem_to_solve

#### **8.0 Iterative Prompt Engineering & Refinement**
- No prompt can be perfect initialy
- Prompt Engineering
    - Build a prompt 
    - Feed it to the model
    - Observe & analyze the output
    - Reiterate to make the prompt better

#### **9.0 Text Summarization & Expansion**


